# Step 7C — Evaluate the supplied sentiment

This notebook records the final confirmed review batch and measures agreement between our manual labels and the dataset's overall polarity labels.

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.metrics import cohen_kappa_score, confusion_matrix

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
REVIEW_PATH = PROJECT_ROOT / "data" / "processed" / "apple_sentiment_manual_review.csv"
review = pd.read_csv(REVIEW_PATH)

In [2]:
final_labels = {
    26: ("Relevant", "Positive", "Apple ranks highly among AI stocks and is presented as a potential buy."),
    27: ("Partly relevant", "Neutral", "Apple is one of several featured companies; no Apple-specific evaluation is provided."),
    28: ("Relevant", "Neutral", "Factual policy change without a clear favorable or unfavorable judgment."),
    29: ("Relevant", "Negative", "Labor-board complaint alleges illegal rules that violated employee rights."),
    30: ("Partly relevant", "Neutral", "Apple is a platform operator in a third-party app-removal request."),
}
for review_id, (relevance, sentiment, notes) in final_labels.items():
    mask = review["review_id"].eq(review_id)
    review.loc[mask, ["manual_relevance", "manual_sentiment", "review_notes"]] = [relevance, sentiment, notes]

assert review[["manual_relevance", "manual_sentiment"]].notna().all().all()
review.to_csv(REVIEW_PATH, index=False)
print("All 30 confirmed reviews are recorded.")

All 30 confirmed reviews are recorded.


In [3]:
review["agrees"] = review["dataset_label"].eq(review["manual_sentiment"])
summary = pd.Series({
    "Reviewed articles": len(review),
    "Exact agreements": int(review["agrees"].sum()),
    "Exact agreement rate": review["agrees"].mean(),
    "Cohen's kappa": cohen_kappa_score(review["manual_sentiment"], review["dataset_label"]),
})
summary

Reviewed articles       30.000000
Exact agreements        12.000000
Exact agreement rate     0.400000
Cohen's kappa            0.217391
dtype: float64

In [4]:
relevance_summary = review.groupby("manual_relevance").agg(
    articles=("review_id", "size"),
    sentiment_agreement=("agrees", "mean"),
)
relevance_summary

,articles,sentiment_agreement
manual_relevance,,
Irrelevant,1,1.000000
Partly relevant,7,0.285714
Relevant,22,0.409091


In [5]:
labels = ["Negative", "Neutral", "Positive"]
comparison = pd.DataFrame(
    confusion_matrix(review["manual_sentiment"], review["dataset_label"], labels=labels),
    index=[f"Manual: {label}" for label in labels],
    columns=[f"Dataset: {label}" for label in labels],
)
comparison

,Dataset: Negative,Dataset: Neutral,Dataset: Positive
Manual: Negative,3,0,5
Manual: Neutral,1,4,12
Manual: Positive,0,0,5


In [6]:
review.loc[~review["agrees"], [
    "review_id", "title", "dataset_label", "manual_sentiment", "manual_relevance", "review_notes"
]].sort_values("review_id")

,review_id,title,dataset_label,manual_sentiment,manual_relevance,review_notes
3,4,Beyond the iPhone: Here's What May Decide Appl...,Positive,Neutral,Relevant,Analytical discussion of Apple's reliance on s...
4,5,Shares of Apple suppliers fall on reports of C...,Positive,Negative,Relevant,China iPhone restrictions and supplier decline...
5,6,"iPhone 15 launch: Release date, price and new ...",Positive,Neutral,Relevant,Factual product-launch preview without a clear...
7,8,Apple Inc. (NASDAQ:AAPL) is largely controlled...,Positive,Neutral,Relevant,Ownership structure is described without a cle...
8,9,The company that makes your iPhone is expandin...,Positive,Neutral,Partly relevant,Primarily about Foxconn and Nvidia; Apple is c...
9,10,Apple Unveils M3 Processors and New MacBook Pros,Positive,Neutral,Relevant,Product optimism is balanced by analyst cautio...
10,11,How China's economic woes could hurt Apple's s...,Positive,Negative,Relevant,Share decline and weak Chinese-segment perform...
12,13,"UPDATE 1-EU asks Apple, Google to clarify app ...",Negative,Neutral,Relevant,Regulators requested information but the excer...
13,14,"Walgreens, Ford, Micron rise premarket; Apple,...",Positive,Negative,Partly relevant,Broad market roundup specifically identifies A...
14,15,Why I Sold Some Apple Stock to End 2023 and Wh...,Positive,Neutral,Relevant,Strong past performance is balanced by the aut...
